# Building a Local MCP Server

Notebook 01 showed what travels between a client and a server. This notebook switches to the server side: how you turn an existing system into an MCP server. You write ordinary Python functions and register them as tools, resources, or prompts, and the official [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk) generates their schemas and handles every JSON-RPC message for you.

You will inspect Northstar, the server used throughout the repository. It wraps a small ticket, policy, and system dataset with one tool, one resource, and one prompt.

## Learning Objectives

By the end of this notebook, you should be able to:

- Explain how the Python SDK turns decorated functions into MCP tools, resources, and prompts.
- Read the input schema the SDK generates from a function's type hints.
- Call a tool in process and tell a schema validation failure from a failure raised by the tool's own code.

## From Python Functions To MCP Primitives

Building a server with the SDK takes three steps: create an `MCPServer`, write one function per capability, and register each function with a decorator. Here is the Northstar server. It is the same code that `build_operations_server` in [`support/mcp_lesson.py`](support/mcp_lesson.py) runs, so the rest of the notebook can use that helper instead:

In [ ]:
import json

from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import BaseModel

from support import load_operations_data

data = load_operations_data()
server = MCPServer(name="northstar-ops", version="0.1.0")


class EscalationResult(BaseModel):
    ticket_id: str
    policy_id: str
    external_system: str
    next_action: str
    requires_human_approval: bool


def find_row(rows, key, value):
    for row in rows:
        if row[key] == value:
            return row
    raise ValueError(f"No row found for {key}={value}")


@server.tool(description="Create an escalation plan for a support ticket.")
def create_escalation(ticket_id: str) -> EscalationResult:
    try:
        ticket = find_row(data["tickets"], "ticket_id", ticket_id)
    except ValueError as error:
        raise ToolError(str(error)) from error
    policy = find_row(data["policies"], "topic", ticket["topic"])
    return EscalationResult(
        ticket_id=ticket["ticket_id"],
        policy_id=policy["policy_id"],
        external_system=ticket["external_system"],
        next_action=f"Use {ticket['external_system']} and follow {policy['text']}",
        requires_human_approval=ticket["priority"] == "critical",
    )


@server.resource("northstar://systems", mime_type="application/json")
def northstar_systems() -> str:
    return json.dumps(data["systems"], indent=2)


@server.prompt(description="Reusable incident-update prompt.")
def incident_update(ticket_id: str) -> str:
    return (
        "Draft an incident update for the customer. "
        f"Use ticket {ticket_id}, cite policy evidence, and request human "
        "approval before promising remediation timing."
    )

Each part of that code becomes part of the protocol contract a client discovers:

| You write | The SDK turns it into |
|---|---|
| `@server.tool` on a function | An entry in `tools/list`, callable with `tools/call` |
| The parameter `ticket_id: str` | An `inputSchema` with a required string `ticket_id` |
| The return type `-> EscalationResult` | An `outputSchema`, and `structuredContent` in every result |
| `description=...` | The description the model reads to decide when to call the tool |
| `raise ToolError(...)` | A result with `isError: true` |
| `@server.resource("northstar://systems")` | An entry in `resources/list`, readable with `resources/read` |
| `@server.prompt` | An entry in `prompts/list`, fetched with `prompts/get` |

You never write a JSON Schema or a JSON-RPC handler by hand. That is why type hints and descriptions matter: they are exactly what the client, and through it the model, gets to see.

<div align="center">
  <img src="assets/02-northstar-server-contract.png" alt="Northstar MCP server contract" width="800" style="max-width: 100%; height: auto;" />
  <p><em>Northstar MCP server contract. Source: repository-authored diagram based on the local fixtures in <code>data/operations/</code> and the official <a href="https://modelcontextprotocol.io/docs/learn/server-concepts" target="_blank" rel="noopener noreferrer">MCP server concepts</a></em></p>
</div>

The server contract is deliberately small: one controlled action, one read-only context source, and one reusable instruction template. That keeps the protocol mechanics visible before adding richer backend integrations.


## Inspect The Server

The helpers in [`support`](support/mcp_lesson.py) build the server and call it in process, without a client or a transport, so you can inspect it directly. Notebook 03 connects to the same server through a real client. The data comes from local JSON files that stand in for three external systems (a warehouse API, an incident database, and a billing API), so every result is deterministic and no credentials are needed.

In [ ]:
from support import (
    build_operations_server,
    discover_server,
    invoke_tool,
    load_operations_data,
    validate_tool_schema,
)

In [ ]:
data = load_operations_data()
{k: len(v) for k, v in data.items()}

`build_operations_server` registers the three functions shown above. Listing the server's primitives shows what a client would discover:

In [ ]:
server = build_operations_server()
discovery = await discover_server(server)
discovery

The SDK generated the tool's input schema from the type hints, and it declares `ticket_id` as required. A client can read that before it calls the tool:

In [ ]:
schema_check = await validate_tool_schema(server, "create_escalation", {"ticket_id"})
schema_check

Invoke the tool directly to inspect the escalation plan. It does not create a ticket or contact an external service.

In [ ]:
result = await invoke_tool(server, "create_escalation", {"ticket_id": "MCP-1001"})
result

## Two Ways A Tool Call Can Fail

A tool call can fail in two places, and as a server author you write the code for only one of them:

- **Before your function runs.** The SDK validates the arguments against the generated input schema. A call without `ticket_id` never reaches `create_escalation`.
- **Inside your function.** The arguments are valid, but your code finds a problem, such as a ticket that does not exist. Raise `ToolError` with a message the model can act on.

Called in process, as in this notebook, both failures raise `ToolError`. Over a real connection, the SDK turns both into a normal result with `isError: true`, the tool error from notebook 01, so the model can read the message and try again.

## Exercise: Find Where Each Failure Comes From

Use `validate_tool_schema` to confirm that `ticket_id` is required and `customer_id` is not. Then call `create_escalation` with `invoke_tool` three times: with the known ticket `MCP-1001`, with no arguments, and with the unknown ticket `MCP-9999`. Catch `ToolError` for the expected failures, without substituting a successful plan, and record whether each call succeeded along with its error message.

Use the messages to decide which failure the SDK raised while checking the schema, and which one `create_escalation` raised itself.

In [ ]:
# @TODO: Check the schema, then compare a known ticket, missing arguments, and MCP-9999.

<details>
<summary>Show Reference Solution</summary>

```python
assert (await validate_tool_schema(server, "create_escalation", {"ticket_id"}))["has_required_fields"]
assert not (await validate_tool_schema(server, "create_escalation", {"customer_id"}))["has_required_fields"]


async def check_escalation(arguments):
    try:
        return {"ok": True, "result": await invoke_tool(server, "create_escalation", arguments)}
    except ToolError as error:
        return {"ok": False, "error": str(error)}


contract_results = [
    await check_escalation(arguments)
    for arguments in [{"ticket_id": "MCP-1001"}, {}, {"ticket_id": "MCP-9999"}]
]
assert [row["ok"] for row in contract_results] == [True, False, False]
contract_results
```

</details>

The missing-argument error reports a `Field required` validation error for `create_escalationArguments`: the SDK checked the call against the generated schema and never ran the function. The `MCP-9999` error says `No row found`: the arguments were valid, so the function ran and raised `ToolError` itself. Over a real connection, both reach the client as `isError: true`.


## Check Your Understanding

1. Why is a tool with typed, required arguments easier for a model to call correctly than one that accepts a free-form request?
2. When you build a server with the Python SDK, where does a tool's input schema come from?
3. A tool cannot find the record the caller asked for. Should it return an empty result, cause a protocol error, or report a tool error?

<details>
<summary>Show Reference Answers</summary>

1. The schema tells the model exactly which arguments exist and what type each must be, and it lets the server reject a malformed call before any backend code runs. A free-form request has to be interpreted, so mistakes surface late or not at all.
2. From the function signature: the parameter names and type hints become the `inputSchema`, and parameters without a default value are required. The return type becomes the `outputSchema`.
3. A tool error: raise `ToolError` with a clear message, so the client receives `isError: true` and the model can correct itself, for example by asking for another ID. An empty result looks like success, and a protocol error signals a broken request, which this is not.

</details>

## Further Reading

- [Build an MCP Server](https://modelcontextprotocol.io/docs/develop/build-server): The official tutorial that builds a weather server with the Python SDK and connects it to a host.
- [MCP Resources](https://modelcontextprotocol.io/specification/2026-07-28/server/resources) and [MCP Prompts](https://modelcontextprotocol.io/specification/2026-07-28/server/prompts): The two primitives this notebook registers next to the tool.